<a href="https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** one row = one page, for one client, on one day — the grain of `fact_content_daily_performance` is `report_date` × `client_hash_id` × `content_hash_id`.

**Time window:** I develop on a single mid-panel month, `month=2026-03` (March 2026). I deliberately avoid the `_sample` table (June 2026, the final month), because the last month is the natural outcome window for any past→future label — developing there would mean peeking at the future I'm supposed to predict.

**Tables:** `fact_content_daily_performance` (daily signals) as the primary table; `dim_content` (one row per page: `word_count`, `content_type`, dates) joined on `content_hash_id` for page metadata.

I verify this grain with a query below rather than assuming it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Setup: DuckDB + Hugging Face connection ---
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')   # достаём токен из Secrets (не виден в коде)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Тест доступа: считаем строки за март 2026 (mid-panel month)
test = con.sql(f"""
    SELECT COUNT(*) AS rows_march
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(test)

# Вариант А — один parquet-файл без папки
try:
    t = con.sql(f"SELECT * FROM read_parquet('{rel}/dim_content.parquet') LIMIT 3").df()
    print("Вариант А сработал:", t.columns.tolist())
except Exception as e:
    print("А не сработал:", str(e)[:100])

# CLAIM: one row = report_date × client × content (no duplicates)
rel = "hf://datasets/FlyRank/internship-warehouse"
fact = f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')"

grain_check = con.sql(f"""
    SELECT
        COUNT(*)                                                   AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_keys
    FROM {fact}
""").df()
print(grain_check)
print("Grain holds if total_rows == unique_keys")

   rows_march
0     9841378
Вариант А сработал: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_keys
0     9841378      9841378
Grain holds if total_rows == unique_keys


## 2. Fields: feature / label / context / excluded

Every field I plan to touch, sorted into four buckets:

**Features (knowable at the decision moment):**
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` — daily search-performance signals.
- Derived CTR = clicks / impressions (built in code, not a raw column).
- `word_count`, `content_type` (from `dim_content`) — page metadata, stable before the decision.
- `content_age_days` = decision_date − `content_created_date`.

**Label / proxy:**
- A future-window decline label I will define later (features from a prior window → decline over a later window). For this contract I only *sketch* it; I do not model yet.

**Context (used for grouping, checks, and joins — never as features):**
- `client_hash_id`, `content_hash_id` — join keys and group keys.
- `report_date`, `month` — time positioning.
- `gsc_data_available`, `ga4_data_available` — availability flags.

**Excluded (with reason):**
- `ga4_*` and `sessions_ai` / `ai_*` — mostly missing for GSC-only clients (sparse); including them would inject NULLs and, for AI, near-empty signal (lane guide warns AI sessions are ~30k of ~79M rows).
- Any future-window field once the label is defined — excluded from features to prevent leakage.
- Raw-origin hashes (`keyword_hash_id`, `url_hash_id`) — grouping only, never decoded.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

Each contract claim gets a query. Then I build a 5-feature frame for March, and finally I spring the leakage trap on purpose.

**Five features, each knowable at the decision moment:**

1. `f_impressions` (gsc_impressions) — measured on the day itself, before any refresh decision. Knowable.
2. `f_clicks` (gsc_clicks) — same day's observed clicks, pre-decision. Knowable.
3. `f_avg_position` (gsc_avg_position) — the day's average ranking position, observed. Knowable.
4. `f_ctr` (clicks / impressions) — derived only from same-day observed columns, no future data. Knowable.
5. `f_has_min_volume` (impressions ≥ 10) — a noise filter computed from the day's impressions. Knowable.

All five come from the *feature window* only — none uses any future outcome.


**The leakage lesson:** the leaky model scores near-perfect only because `ctr` is exactly what the label was derived from — the model reads the answer off its own input. This is why any label-source column must be excluded from features. I keep the honest number and drop `ctr` from the feature set for this label.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# QUERY 1: row count and date span of my March slice
span = con.sql(f"""
    SELECT
        COUNT(*)          AS rows_march,
        MIN(report_date)  AS first_day,
        MAX(report_date)  AS last_day,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        COUNT(DISTINCT client_hash_id)  AS unique_clients
    FROM {fact}
""").df()
print(span)

# QUERY 2: availability — how many rows have real GSC vs GA4 data
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows
    FROM {fact}
""").df()
print(avail)
print("Note: ga4_rows << gsc_rows -> many pages are GSC-only. GA4 features would be mostly NULL.")

# QUERY 3: build a 5-feature frame for March (one row = page-day)
features = con.sql(f"""
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions                                    AS f_impressions,
        gsc_clicks                                         AS f_clicks,
        gsc_avg_position                                   AS f_avg_position,
        CASE WHEN gsc_impressions > 0
             THEN gsc_clicks * 1.0 / gsc_impressions
             ELSE 0 END                                    AS f_ctr,
        CASE WHEN gsc_impressions >= 10 THEN 1 ELSE 0 END  AS f_has_min_volume
    FROM {fact}
    WHERE gsc_data_available IS TRUE
    LIMIT 1000
""").df()
print(features.head())
print(f"\nFeature frame shape: {features.shape}")


# THE TRAP: leakage demonstrated on a BALANCED label
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

# Pull a slice with real signal (pages that actually got impressions)
df = con.sql(f"""
    SELECT
        gsc_impressions   AS impressions,
        gsc_clicks        AS clicks,
        gsc_avg_position  AS avg_position,
        CASE WHEN gsc_impressions > 0
             THEN gsc_clicks * 1.0 / gsc_impressions ELSE 0 END AS ctr
    FROM {fact}
    WHERE gsc_data_available IS TRUE AND gsc_impressions >= 10
    LIMIT 20000
""").df()

# Balanced label: is this page in the WORSE half by position?
# (median split guarantees ~50/50 classes, so guessing is not trivial)
median_pos = df["avg_position"].median()
df["label"] = (df["avg_position"] > median_pos).astype(int)

print(f"Median position used as split: {median_pos:.2f}")
print("Label balance:", df["label"].value_counts().to_dict())
print(f"Positive rate: {df['label'].mean():.1%}\n")

y = df["label"]

# HONEST features: impressions + clicks + ctr (NOT position)
X_honest = df[["impressions", "clicks", "ctr"]]
honest = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=42),
                         X_honest, y, cv=3, scoring="accuracy").mean()

# LEAKY features: add avg_position — the column the label was DERIVED from
X_leaky = df[["impressions", "clicks", "ctr", "avg_position"]]
leaky = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=42),
                        X_leaky, y, cv=3, scoring="accuracy").mean()

print(f"Honest accuracy (impressions, clicks, ctr):     {honest:.3f}")
print(f"Leaky accuracy  (+ avg_position, label source): {leaky:.3f}")
print(f"\nThe leak inflates accuracy by {(leaky-honest):.3f} — because the label IS derived from avg_position.")
print("Keeping the honest number; avg_position is removed from features for this label.")

   rows_march  first_day   last_day  unique_pages  unique_clients
0     9841378 2026-03-01 2026-03-31        331437              55
   total_rows  gsc_rows  ga4_rows
0     9841378   3611061    413966
Note: ga4_rows << gsc_rows -> many pages are GSC-only. GA4 features would be mostly NULL.
            content_hash_id report_date  f_impressions  f_clicks  \
0  content_b7e512995f79d5a6  2026-03-01             20         0   
1  content_05597932fe4da067  2026-03-01              1         0   
2  content_7a105f548d9c6916  2026-03-01            125         1   
3  content_905aa32a0230694e  2026-03-01              7         0   
4  content_a3ea9792f793ec72  2026-03-01             11         0   

   f_avg_position  f_ctr  f_has_min_volume  
0        3.350000  0.000                 1  
1        0.000000  0.000                 0  
2        4.928000  0.008                 1  
3        4.000000  0.000                 0  
4        2.272727  0.000                 1  

Feature frame shape: (1000, 7)

## 4. Data limits

**What this data can never tell me / where it's limited:**

1. **Unbalanced panel.** Per `dim_clients`, clients start tracking at different dates (`gsc_data_start`, `ga4_data_start` differ; some are null). Different clients contribute different history depths, so any time-window feature must respect each client's own start.

2. **GSC-only early rows.** Many rows have `gsc_data_available IS TRUE` but `ga4_data_available` null/false (the availability query above shows GA4 rows ≪ GSC rows). GA4-based features are mostly missing and can't be treated as "zero traffic".

3. **AI signal is too sparse to model.** `sessions_ai` and `ai_*` are almost entirely NULL in this slice — usable for EDA at most, not for a classifier (lane guide: ~30k AI rows of ~79M).

4. **No causal claims.** This is observational daily data. It can rank candidates for review; it cannot prove a refresh *causes* recovery without an experiment.

5. **Pseudonymized only.** Hash keys are salted; I can group by client/page but never recover real clients, URLs, or queries.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.